In [2]:
from bs4 import BeautifulSoup


def read_cv(path_cv):
    with open(path_cv, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")
    return soup

path_html = 'data/curriculos/2747150211073176/cv.html'
soup = read_cv(path_html)

# Create Profile

In [3]:
from lib.parser.lattes.profile import save_profile


lattes_id = '2747150211073176'
save_profile(soup, lattes_id)

In [4]:
from sqlalchemy.orm import Session, sessionmaker
from lib.db.database import engine

SessionLocal = sessionmaker(
    bind=engine,
    autoflush=False,
    autocommit=False
)
session = SessionLocal()

In [5]:
from lib.db.crud.authors.profile import get_or_create_profile

author_db = get_or_create_profile(session, lattes_id)
author_db

2026-04-23 16:21:06,572 INFO sqlalchemy.engine.Engine SELECT DATABASE()
2026-04-23 16:21:06,573 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-04-23 16:21:06,575 INFO sqlalchemy.engine.Engine SELECT @@sql_mode
2026-04-23 16:21:06,576 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-04-23 16:21:06,577 INFO sqlalchemy.engine.Engine SELECT @@lower_case_table_names
2026-04-23 16:21:06,577 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-04-23 16:21:06,579 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-23 16:21:06,605 INFO sqlalchemy.engine.Engine SELECT authors.id, authors.full_name, authors.given_name, authors.family_name, authors.orcid, authors.lattes_id, authors.is_inpa_researcher, authors.normalized_full_name, authors.canonical_source, authors.needs_review, authors.affiliation_id 
FROM authors 
WHERE authors.lattes_id = %(lattes_id_1)s
2026-04-23 16:21:06,606 INFO sqlalchemy.engine.Engine [generated in 0.00086s] {'lattes_id_1': '2747150211073176'}
2026-04-23 16:21:06,619 INFO sq

In [6]:
author_db.full_name

2026-04-23 16:21:13,572 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-23 16:21:13,574 INFO sqlalchemy.engine.Engine SELECT authors.id AS authors_id, authors.full_name AS authors_full_name, authors.given_name AS authors_given_name, authors.family_name AS authors_family_name, authors.orcid AS authors_orcid, authors.lattes_id AS authors_lattes_id, authors.is_inpa_researcher AS authors_is_inpa_researcher, authors.normalized_full_name AS authors_normalized_full_name, authors.canonical_source AS authors_canonical_source, authors.needs_review AS authors_needs_review, authors.affiliation_id AS authors_affiliation_id 
FROM authors 
WHERE authors.id = %(pk_1)s
2026-04-23 16:21:13,575 INFO sqlalchemy.engine.Engine [generated in 0.00079s] {'pk_1': 1}


'Adalberto Luis Val'

# Atualiza Curiculos

In [1]:
from lib.db.make_session import local_session
from lib.updates.check_cv import check_cv_update

In [2]:
from lib.scrap_lattes.driver import make_driver


driver = make_driver(headless=False)

In [3]:
from lib.scrap_lattes.baixar import baixar_lattes

lattes_id = '2747150211073176'  
html = baixar_lattes(driver, lattes_id)

Baixando curriculo com ID: 2747150211073176


In [4]:
with open('data/curriculo/2747150211073176/cv.html', 'w') as f:
    f.write(html)

In [20]:
session = local_session()
lattes_id = '2747150211073176'  
html, update = check_cv_update(session, lattes_id)

2026-04-23 15:02:32,899 INFO sqlalchemy.engine.Engine SELECT DATABASE()
2026-04-23 15:02:32,903 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-04-23 15:02:32,911 INFO sqlalchemy.engine.Engine SELECT @@sql_mode
2026-04-23 15:02:32,912 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-04-23 15:02:32,913 INFO sqlalchemy.engine.Engine SELECT @@lower_case_table_names
2026-04-23 15:02:32,914 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-04-23 15:02:32,917 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-23 15:02:33,035 INFO sqlalchemy.engine.Engine SELECT lattes.id, lattes.author_id, lattes.lattes_id, lattes.lattes_update, lattes.html, lattes.updated_at 
FROM lattes 
WHERE lattes.lattes_id = %(lattes_id_1)s
2026-04-23 15:02:33,036 INFO sqlalchemy.engine.Engine [generated in 0.00173s] {'lattes_id_1': '2747150211073176'}
CV não encontrado


In [8]:
from lib.parser.lattes.artigos_completos import get_artigos_completos, slipt_artigos
list_artigos = get_artigos_completos(soup)

In [9]:
len(list_artigos)

297

In [14]:
def get_autores(raw_artigo):
    autores = []
    for tag in raw_artigo.find_all(["a", "b"]):
        name = tag.get_text(strip=True)
        if "," in name and any(c.isupper() for c in name):
            d_a = {'name': name}
            href = tag.attrs.get('href')
            if href:
                id_lattes = href.split('/')[-1]
                d_a['id_lattes'] = id_lattes
            autores.append(d_a)
    return autores

In [10]:
c_doi, s_doi = slipt_artigos(list_artigos)
len(c_doi)

261

# Artigos com doi

In [ ]:
from lib.crossref.get import get_article_crossref


error = get_article_crossref(c_doi, lattes_id)

In [14]:
error

[{'doi': ['10.11111/jfb.70021'],
  'issn': ['00221112'],
  'volume': ['00'],
  'issue': [''],
  'paginaInicial': ['00'],
  'titulo': ['Plastic contamination in fish digestive tracts in Amazonian rivers during a period of extreme low water'],
  'sequencial': ['15'],
  'nomePeriodico': ['JOURNAL OF FISH BIOLOGY']},
 {'doi': ['10.14201/reb20231021'],
  'issn': ['23864540'],
  'volume': ['10'],
  'issue': [''],
  'paginaInicial': ['00'],
  'titulo': ['Para desenvolver bionegócios na Amazônia'],
  'sequencial': ['17'],
  'nomePeriodico': ['REVISTA DE ESTUDIOS BRASILEÑOS']},
 {'doi': ['10.1016 / j.scitotenv.2020.138628'],
  'issn': ['00489697'],
  'volume': ['726'],
  'issue': [''],
  'paginaInicial': ['138628'],
  'titulo': ['Extreme climate scenario and parasitism affect the Amazonian fish Colossoma macropomum'],
  'sequencial': ['95'],
  'nomePeriodico': ['SCIENCE OF THE TOTAL ENVIRONMENT']}]

In [ ]:
error = []
with open("data/artigos/val.jsonl", "w", encoding="utf-8") as f:
    
    for i in c_doi:
        doi = i['doi'][0]
        url = f"https://api.crossref.org/v1/works/{doi}"
        r = httpx.get(url)
        print(r.status_code)
        if r.status_code == 200:
            item = r.json()['message']
            json.dump(item, f)
            f.write("\n")
        else:
            print(f"Error fetching data for DOI: {doi}, status code: {r.status_code}")
            error.append(i)